In [ ]:
import json
import polars as pl
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

## Process mave db metadata

In [ ]:
with open('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/main.json') as f:
    d = json.load(f)

d

In [ ]:
d['experimentSets'][1]

In [ ]:
d['experimentSets'][0]['experiments'][0]['scoreSets'][0]

In [ ]:
d['experimentSets'][1]['experiments'][0]['scoreSets'][0]

In [ ]:
import pandas as pd

# Suppose your full list is called `data`
# Example: data = [ {...}, {...}, ... ]

rows = []

for item in d['experimentSets']:
    urn = item.get('urn')
    published_date = item.get('publishedDate')
    set_id = item.get('id')
    record_type = item.get('recordType')

    # Loop over experiments in this experiment set
    for exp in item.get('experiments', []):
        exp_title = exp.get('title')
        exp_short_desc = exp.get('shortDescription')
        exp_abstract = exp.get('abstractText')
        exp_method = exp.get('methodText')
        exp_urn = exp.get('urn')
        exp_creation_date = exp.get('creationDate')

        # Loop over scoreSets
        for score in exp.get('scoreSets', []):
            score_title = score.get('title')
            num_variants = score.get('numVariants')
            score_urn = score.get('urn')
            license_name = score.get('license', {}).get('longName')
            
            target_genes = score.get('targetGenes', [])
            # Loop over target genes
            for gene in target_genes:
                gene_name = gene.get('name')
                category = gene.get('category')

                organism_name = None
                target_sequence = gene.get('targetSequence')
                if target_sequence:
                    taxonomy = target_sequence.get('taxonomy')
                    if taxonomy:
                        organism_name = taxonomy.get('organismName')
            
                # Extract external IDs
                ensembl_id = None
                refseq_id = None
                uniprot_id = None

                for ext_id in gene.get('externalIdentifiers', []):
                    identifier = ext_id.get('identifier', {})
                    db_name = identifier.get('dbName', '').lower()
                    id_value = identifier.get('identifier')

                    if db_name == 'ensembl':
                        ensembl_id = id_value
                    elif db_name == 'refseq':
                        refseq_id = id_value
                    elif db_name == 'uniprot':
                        uniprot_id = id_value

                rows.append({
                    'set_urn': urn,
                    'set_published_date': published_date,
                    'set_id': set_id,
                    'record_type': record_type,

                    'experiment_title': exp_title,
                    'experiment_short_desc': exp_short_desc,
                    'experiment_abstract': exp_abstract,
                    'experiment_method': exp_method,
                    'experiment_urn': exp_urn,
                    'experiment_creation_date': exp_creation_date,

                    'score_title': score_title,
                    'num_variants': num_variants,
                    'score_urn': score_urn,
                    'license_name': license_name,

                    'target_gene': gene_name,
                    'category': category,
                    'ensembl_id': ensembl_id,
                    'refseq_id': refseq_id,
                    'uniprot_id': uniprot_id,
                    'organism_name': organism_name
                })

# Build DataFrame
mave_db = pl.DataFrame(rows)

# Extract gene symbols
# mave_db = mave_db.with_columns(
#     pl.col('target_gene').str.split(' ').list.get(0).alias('gene_symbol')
# )

mave_db = mave_db.with_columns(
    pl.col('target_gene')
    .str.split(' ')
    .list.get(0)
    .str.to_uppercase()
    .alias('gene_symbol')
)

mave_db

In [ ]:
mave_db['category'].value_counts().sort('count', descending=True)

In [ ]:
mave_db['organism_name'].value_counts().sort('count', descending=True)

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['target_gene'].value_counts().sort('count', descending=True)

In [ ]:
# Merge to get ENSEMBLE gene ids

dgid = pl.read_parquet('/s/project/deeprvat/deeprvat_input/protein_coding_genes.parquet').rename({'gene_name':'gene_symbol'})

dgid = dgid.with_columns(
    pl.col('gene').str.split('.').list.get(0).alias('gene_id')
).drop(['__index_level_0__', 'gene_type', 'id', 'gene'])

dgid

In [ ]:
mave_db = mave_db.join(dgid, on='gene_symbol', how='left')
mave_db

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['gene_id'].value_counts().sort('count', descending=True)

In [ ]:
tmp = mave_db.filter(pl.col('organism_name')=='Homo sapiens').filter(pl.col('category')=='protein_coding').filter(pl.col('gene_id').is_null())['target_gene', 'gene_symbol', 'set_urn'].unique().sort(by='target_gene')
tmp

In [ ]:
# Dictionary to map the ambiguous target genes

map_dict = {
    'AID': 'AICDA',
    'ARK2C Zinc finger, RING-type domain': 'ARK2C',
    'Aβ42': 'APP',
    'COMT_ROI1_2': 'COMT',
    'DUX4': 'DUX4',
    'GB1': 'IGBP1',
    'GRLF1 FF domain': 'ARHGAP35',
    'Glycophorin A': 'GYPA',
    'IGHG1': 'IGHG1',
    'NA Transcription factor IIS, N-terminal domain': 'TCEA1',
    'NA Ubiquitin-like domain': 'UBL3',
    'PSD95 PDZ3': 'DLG4',
    'RAF': 'RAF1',
    'Ras': 'KRAS',
    'S505N MPL': 'MPL',
    'S505N MPL': 'MPL',
    'SMN Tudor domain': 'SMN',
    'VKOR': 'VKORC1',
    'W515K MPL': 'MPL',
    'alpha-synuclein': 'SNCA',
    'hYAP65 WW domain': 'YAP65',
    'human L-Selectin': 'CD62L',
    'p53': 'TP53',
}

# Convert to pandas
df_pd = mave_db.filter(pl.col('target_gene').is_in(map_dict.keys())).to_pandas()

# Map only if key in map_dict, else keep original value
df_pd['gene_symbol'] = df_pd['target_gene'].apply(
    lambda x: map_dict[x]
)

df_pd['gene_symbol'].value_counts().sort_values(ascending=False)

In [ ]:
# Append back to mave_db
sub_mave_db = mave_db.filter(~pl.col('target_gene').is_in(map_dict.keys()))

mave_db = pl.concat([sub_mave_db, pl.from_pandas(df_pd)])
mave_db

In [ ]:
mave_db = mave_db.drop('gene_id').join(dgid, on='gene_symbol', how='left')
mave_db

In [ ]:
mave_db.filter(pl.col('organism_name')=='Homo sapiens')['gene_id'].value_counts().sort('count', descending=True)

## Check gene intersection

In [ ]:
gb_res = pd.read_parquet('/s/project/deeprvat/ukb_gym/genebass/genebass_all_associations_p_e-5.parquet')

pval_cutoffs = {'burden': 6.7e-7, 'skato': 2.5e-7} # pvalue thresholds used in genebass paper
mask = (gb_res['Pvalue'] < pval_cutoffs['skato'])
mask |= (gb_res['Pvalue_Burden'] < pval_cutoffs['burden'])
gb_res["significant"] = mask

gb_res = gb_res.query("(significant == True) & (trait_type=='continuous') & (modifier != 'custom') & ('pLoF' in annotation)")
gb_res

In [ ]:
gb_res['gene_id'].unique()

In [ ]:
mgb_genes = mave_db.filter(pl.col('gene_id').is_in(gb_res['gene_id'].unique()))['gene_symbol', 'gene_id'].unique()
mgb_genes

## Check intersection of variants

### Consolidate all the MAVE scores

In [ ]:
mave_filt = mave_db.filter(pl.col('gene_symbol').is_in(mgb_genes['gene_symbol'])).sort('num_variants', descending=True)
mave_filt

In [ ]:
mave_filt['num_variants'].sum()

In [ ]:
mave_dir = '/s/project/deeprvat/ukb_gym/experimental_assays/mave_db'
a = mave_filt['score_urn'][10].replace(":", "-")

s = pl.read_csv(f"{mave_dir}/csv/{a}.scores.csv").with_columns(
    pl.col('score').cast(pl.Float64, strict=False)
)
s

In [ ]:
mave_dir = '/s/project/deeprvat/ukb_gym/experimental_assays/mave_db'
a = mave_filt['score_urn'][1].replace(":", "-")

s = pl.read_csv(f"{mave_dir}/csv/{a}.scores.csv").with_columns(
    pl.col('score').cast(pl.Float64, strict=False)
)
s

In [ ]:
plt.hist(s['score'], bins=100)
plt.show()

In [ ]:
mave_filt['score_urn', 'gene_id', 'gene_symbol', 'target_gene'].unique()

In [ ]:
mave_dir = '/s/project/deeprvat/ukb_gym/experimental_assays/mave_db'
id_cols = ['score_urn', 'gene_id', 'gene_symbol', 'target_gene']

# Define the core columns to select from each score CSV
core_cols = ['accession', 'hgvs_nt', 'hgvs_splice', 'hgvs_pro', 'score']

var_score_list = []

req_mave_df = mave_filt.select(id_cols).unique()

for row in tqdm(req_mave_df.iter_rows(named=True), total=req_mave_df.height):
    score_urn_safe = row['score_urn'].replace(":", "-")
    csv_path = f"{mave_dir}/csv/{score_urn_safe}.scores.csv"

    tmp = pl.read_csv(csv_path, null_values="NA")

    # Find possible extra column: anything not in core
    extra_cols = [col for col in tmp.columns if col not in core_cols]

    if extra_cols:
        raw_extra_col = extra_cols[0]
        # Sanitize: lower, replace - or : with _, strip whitespace
        extra_col_clean = (
            raw_extra_col.lower()
            .replace("-", "_")
            .replace(":", "_")
            .strip()
        )
    else:
        raw_extra_col = None
        extra_col_clean = None

    if raw_extra_col:
        tmp = tmp.with_columns([
            pl.col(raw_extra_col).alias('extra_col_value'),
            pl.lit(extra_col_clean).alias('extra_col_type')
        ])
    else:
        tmp = tmp.with_columns([
            pl.lit(None).alias('extra_col_value'),
            pl.lit(None).alias('extra_col_type')
        ])

    tmp = tmp.with_columns(
        pl.col('score').cast(pl.Float32, strict=False),
        pl.col('extra_col_value').cast(pl.Float32, strict=False),
        pl.lit(row['score_urn']).alias('score_urn'),
        pl.lit(row['gene_id']).alias('gene_id'),
        pl.lit(row['gene_symbol']).alias('gene_symbol'),
        pl.lit(row['target_gene']).alias('target_gene')
    ).select(
        id_cols + core_cols + ['extra_col_type', 'extra_col_value']
    )

    var_score_list.append(tmp)

mave_var_scores = pl.concat(var_score_list)
# mave_var_scores.write_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/mave_scores_genebass_genes.parquet')
mave_var_scores

In [ ]:
mave_dir = '/s/project/deeprvat/ukb_gym/experimental_assays/mave_db'
a = mave_filt['score_urn'][10].replace(":", "-")

s = pl.read_csv(f"{mave_dir}/csv/{a}.scores.csv").with_columns(
    pl.col('score').cast(pl.Float64, strict=False),
)

### Amino acid mapping

In [ ]:
mave_var_scores = pl.read_parquet('/s/project/deeprvat/ukb_gym/experimental_assays/mave_db/mave_scores_genebass_genes.parquet')
mave_var_scores

In [ ]:
aa_3to1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D",
    "Cys": "C", "Gln": "Q", "Glu": "E", "Gly": "G",
    "His": "H", "Ile": "I", "Leu": "L", "Lys": "K",
    "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S",
    "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V",
    "Ter": "*"   # Sometimes for stop codon
}

In [ ]:
# Example DataFrame
df = pl.DataFrame({
    "vep_protein_change": ["p.Val71Ala", "p.Glu23Ter", "p.Lys100Arg"]
})

# Extract parts
df = df.with_columns([
    pl.col("vep_protein_change").str.extract(r"p\.([A-Za-z]+)", 1).alias("ref_aa_3"),
    pl.col("vep_protein_change").str.extract(r"p\.[A-Za-z]+(\d+)", 1).alias("pos"),
    pl.col("vep_protein_change").str.extract(r"p\.[A-Za-z]+\d+([A-Za-z]+)", 1).alias("alt_aa_3"),
])

# Map 3-letter to 1-letter using map_elements
df = df.with_columns([
    pl.col("ref_aa_3").map_elements(lambda aa: aa_3to1.get(aa, aa)).alias("ref_aa"),
    pl.col("alt_aa_3").map_elements(lambda aa: aa_3to1.get(aa, aa)).alias("alt_aa"),
])

# Recombine to single-letter notation
df = df.with_columns([
    pl.format("{}{}{}", pl.col("ref_aa"), pl.col("pos"), pl.col("alt_aa")).alias("mutant_short")
])

print(df)


### merge with RAP variants

In [ ]:
# rap_vars = rap_vars.with_columns([
#     # Split `protein_position` on '/' and take the first part
#     pl.col("protein_position").str.split("/").list.get(0).alias("mutant_position"),
#     pl.col("protein_position").str.split("/").list.get(1).alias("protein_length"),
# ])

# rap_vars = rap_vars.with_columns([
#     # Replace '/' in `Amino_acids` with `mutant_position`
#     (
#         pl.format(
#             "{}{}{}",
#             pl.col("amino_acids").str.split("/").list.get(0),
#             pl.col("mutant_position"),
#             pl.col("amino_acids").str.split("/").list.get(1)
#         )
#     ).alias("mutant")
# ])

# rap_vars['mutant_position'] = rap_vars['protein_position'].str.split('/').str[0]
# rap_vars['mutant'] = rap_vars.apply(lambda row: row['Amino_acids'].replace('/', str(row['mutant_position'])), axis=1)

# rap_vars.write_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass_1e6_coding_variants.parquet')

rap_vars = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/genebass_1e6_coding_variants.parquet')
rap_vars

## Filter out if abstract has words:

- yeast
- bacteria